# Explore the interim tables

Runs after `mammo flatten`, before `features/build_train_set.py` is written.
Column names below assume the output of `fhir/flatten.py` and will need to be
reconciled with whatever it actually produces.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

from mammography.io import read_table

patients = read_table("interim", "patients")
comms = read_table("interim", "communications", parse_dates=["sent_date"])
appts = read_table("interim", "appointments", parse_dates=["start"])

## 1. Null rate per column

Decides which planned features actually exist. A column that is mostly empty,
or only populated after some year, is not a feature.

In [ ]:
for name, df in [("patients", patients), ("communications", comms), ("appointments", appts)]:
    print(f"\n{name}  {df.shape}")
    print(df.isna().mean().sort_values(ascending=False).to_string())

## 2. Days from invitation to mammography

Decides `attendance_window_days` for the label. Every invitation is paired with
every later appointment of the same patient on purpose — the point is the shape
of the delay distribution, not a matched join. Read the cutoff off the plot:
where the bulk ends, the rest is attendance for unrelated reasons.

In [ ]:
pairs = comms[["patient_id", "sent_date"]].merge(
    appts[["patient_id", "start"]], on="patient_id", how="inner"
)
pairs["days_to_appointment"] = (pairs["start"] - pairs["sent_date"]).dt.days
after = pairs.loc[pairs["days_to_appointment"].between(0, 730), "days_to_appointment"]

after.hist(bins=73)
plt.xlabel("days from invitation to mammography")
plt.ylabel("appointments")
plt.show()

after.describe()